# harmonic_practice — Reinforcement Learning Interview Prep

**Interview:** 55-minute RL-focused call. Conceptual discussion + implementation + optimization.
**Required reading:** [Spinning Up in Deep RL](https://spinningup.openai.com/en/latest/)
**Company focus:** RL for solving formal math problems at competition and research level.

**Question sources:** wecreateproblems (100+ RL bank) · interviewnode RL guide · Spinning Up (OpenAI) · aiinterviewprep.substack.com scenario traps · index.dev Top 50 RL engineer questions

**How to use:**
- Read the "Appears in" line — these are verbatim or near-verbatim real interview questions.
- Answer the **PREDICT** section out loud before running any code. This is what the interviewer evaluates.
- Target ~6 minutes per question.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## Q1 — Define the MDP tuple and implement the Bellman equation

**Appears in:** "What is a Markov Decision Process? Define the full tuple." (wecreateproblems, interviewnode, climbtheladder — **near-universal, asked in virtually every RL screen**). "What is the Bellman equation? Write it out." (climbtheladder, wecreateproblems, index.dev).

**Why it matters for formal math RL:**
A theorem prover is an MDP. The *state* is the current proof state (what's been proven, what's left). The *action* is the next tactic to apply. The *reward* is +1 for a completed proof. Before you can design any RL system, you must be able to state this formally. Interviewers open with this to check that you can frame a problem from scratch.

**The MDP tuple: (S, A, P, R, γ)**
- **S**: state space
- **A**: action space
- **P(s'|s,a)**: transition dynamics (probability of landing in s' after action a in state s)
- **R(s,a)**: reward function
- **γ ∈ [0,1)**: discount factor

**The Bellman equation** (the most important equation in RL — memorize this):
```
V^π(s) = Σ_a π(a|s) · [ R(s,a) + γ · Σ_s' P(s'|s,a) · V^π(s') ]
```
For the *optimal* value function (Bellman optimality equation):
```
V*(s) = max_a [ R(s,a) + γ · Σ_s' P(s'|s,a) · V*(s') ]
```

**TASK:**
You are given a deterministic 4-state GridWorld (states 0→3, goal=3, reward=+1 on reaching goal, γ=0.9).

1. Implement `bellman_backup(V, s, gamma)`: for a single non-terminal state `s`, return `max over actions of [R(s,a) + gamma * V[s']]`.
2. Run **value iteration** — repeat Bellman backups for all states until `max|V_new - V_old| < 1e-6`.
3. Print V* and the optimal policy (argmax action per state).

**PREDICT (say this out loud):**
- What is V*(2)? (one step from goal — derive: R + γ·V*(3) = 1 + 0.9·0 = 1? Why is V*(3) = 0, not 1?)
- Can V*(s) ever exceed 1.0 in this MDP? Why or why not?
- What does the Markov property buy us here — what would break without it?

In [ ]:
def gridworld_step(state, action, n=4, goal=3):
    """Deterministic GridWorld. action: 0=left, 1=right. Returns (next_state, reward)."""
    next_s = np.clip(state + (1 if action == 1 else -1), 0, n - 1)
    reward = 1.0 if next_s == goal else 0.0
    return next_s, reward

#V*(2)= max (r(2, a) + 0.9 *V(s'))
#if a=3: 1+ 0.9(0)= 1
#if a=1: 0+ 0.9(V*(1)) <0.9
#max V*(2)=1
def bellman_backup(V, s, gamma=0.9, goal=3):
    """
    Compute max_a [R(s,a) + gamma * V[s']] for non-terminal state s.
    Returns the updated V(s) value.
    """
    # YOUR CODE HERE
    # For each action in {0, 1}:
    #   next_s, reward = gridworld_step(s, action)
    #   q_value = reward + gamma * V[next_s]
    # Return max q_value
    

def value_iteration(gamma=0.9, tol=1e-6, goal=3, n=4):
    V = np.zeros(n)
    for _ in range(1000):
        V_new = V.copy()
        for s in range(n):
            if s == goal:
                continue          # terminal state: V(goal) = 0 by convention
            V_new[s] = bellman_backup(V, s, gamma, goal)
        if np.max(np.abs(V_new - V)) < tol:
            break
        V = V_new
    return V

V_star = value_iteration()
print("V*(s):", np.round(V_star, 4))
print()

# Derive optimal policy
print("Optimal policy (greedy w.r.t. V*):")
for s in range(4):
    if s == 3:
        print(f"  state {s}: terminal")
        continue
    q_values = [gridworld_step(s, a)[1] + 0.9 * V_star[gridworld_step(s, a)[0]] for a in range(2)]
    best_action = np.argmax(q_values)
    print(f"  state {s}: {'right' if best_action else 'left'}  (Q={np.round(q_values, 4)})")

print()
print("Verification — expected V* = [γ³, γ², γ¹, 0] = [{:.3f}, {:.3f}, {:.3f}, 0]".format(
    0.9**3, 0.9**2, 0.9**1))

V*(s): [nan nan nan  0.]

Optimal policy (greedy w.r.t. V*):
  state 0: left  (Q=[nan nan])
  state 1: left  (Q=[nan nan])
  state 2: left  (Q=[nan  1.])
  state 3: terminal

Verification — expected V* = [γ³, γ², γ¹, 0] = [0.729, 0.810, 0.900, 0]


## Q2 — TD learning vs Monte Carlo: implement both return estimates

**Appears in:** "What is temporal difference (TD) learning? How does it differ from Monte Carlo methods?" (wecreateproblems — very high frequency). "Compare TD(0), TD(λ), and Monte Carlo methods." (index.dev). "What is reward-to-go? Why does it reduce variance vs. using total trajectory return?" (Spinning Up — verbatim from the policy optimization intro).

**Why it matters:**
Spinning Up explicitly introduces **reward-to-go** as a variance-reduction technique you must understand before PPO. The key insight: actions at time t cannot affect rewards at time t-1, so including past rewards in the policy gradient weight only adds noise. This is the first step toward GAE (Q9 below).

**Two ways to estimate returns:**

**Monte Carlo (full-trajectory):**
```
G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ... + γ^{T-t}·r_T
```
Unbiased but high variance. Must wait until episode ends.

**TD(0) (one-step bootstrap):**
```
G_t^{TD} = r_t + γ · V(s_{t+1})
```
Lower variance (V smooths out noise) but biased (V is an approximation). Can update online.

**TASK:**
1. `monte_carlo_returns(rewards, gamma)` — backward recurrence, returns list of G_t
2. `td_targets(rewards, values, gamma)` — one-step TD targets `r_t + gamma * V(s_{t+1})`; use `values[t+1]` for all but the last step (terminal: `r_T + 0`)
3. Run both on the same episode and print the difference. Then show: with a perfect value function, TD targets ≈ MC returns.

**PREDICT (say this out loud):**
- MC returns for `rewards=[0,0,0,1]` with γ=0.9: what is G_0? G_1? G_2?
- TD target at t=2 given perfect `V=[0.729, 0.81, 0.9, 0]`: what is `r_2 + γ·V[3]`?
- Why does TD have lower variance than MC? (think: what is TD bootstrapping from?)
- When would you prefer MC over TD? (hint: think about bias in early training)

In [ ]:
def monte_carlo_returns(rewards, gamma=0.9):
    """
    Reward-to-go: G_t = r_t + gamma*r_{t+1} + ...
    Returns list of length T via backward recurrence.
    """
    # YOUR CODE HERE
    pass

def td_targets(rewards, values, gamma=0.9):
    """
    One-step TD targets: G_t^TD = r_t + gamma * V(s_{t+1})
    rewards: list length T
    values:  list length T+1  (values[T] = 0 for terminal)
    Returns list of TD targets, length T.
    """
    # YOUR CODE HERE
    pass


# --- Same episode, two estimators ---
rewards = [0., 0., 0., 1.]
V_perfect = [0.729, 0.81, 0.9, 0.0, 0.0]   # V*(s) + terminal padding

mc  = monte_carlo_returns(rewards, gamma=0.9)
td  = td_targets(rewards, V_perfect, gamma=0.9)

print(f"{'t':>3}  {'reward':>7}  {'MC return G_t':>14}  {'TD target':>10}  {'difference':>10}")
print("-" * 52)
for t, (r, g, d) in enumerate(zip(rewards, mc, td)):
    print(f"{t:3d}  {r:7.1f}  {g:14.4f}  {d:10.4f}  {abs(g-d):10.6f}")

print()
print("With a perfect value function, TD ≈ MC (differences should be ~0)")
print()

# --- Show variance difference between MC and TD ---
np.random.seed(0)
print("Variance comparison (noisy environment, 100 episodes):")
mc_G0s, td_G0s = [], []
for _ in range(100):
    # Stochastic episode: each step has 80% chance of reward 0, 20% chance of 1
    episode_rewards = [float(np.random.rand() < 0.2) for _ in range(4)]
    mc_G0s.append(monte_carlo_returns(episode_rewards)[0])
    V_approx = [0.5, 0.5, 0.5, 0.5, 0.0]   # imperfect value estimate
    td_G0s.append(td_targets(episode_rewards, V_approx)[0])

print(f"  MC  G_0 variance: {np.var(mc_G0s):.4f}")
print(f"  TD  G_0 variance: {np.var(td_G0s):.4f}  (should be lower)")

## Q3 — The policy gradient theorem and the log-derivative trick

**Appears in:** "What are policy gradient methods? How do they differ from value-based methods?" (wecreateproblems, interviewnode, climbtheladder — **near-universal**). "What is the log-derivative trick? Why is it essential to the policy gradient derivation?" (Spinning Up — verbatim). "What is the REINFORCE algorithm? Describe the update rule." (wecreateproblems, interviewnode).

**Why it matters:**
This is the mathematical core of every modern LLM training system (InstructGPT, DeepSeek-R1, etc.). REINFORCE is not just an algorithm — it's the proof that you can optimize expected reward with gradient descent even when the environment is non-differentiable. The **log-derivative trick** is the key step, and Spinning Up covers it explicitly.

**The log-derivative trick:**
```
∇_θ π_θ(a|s) = π_θ(a|s) · ∇_θ log π_θ(a|s)
```
This lets us write the gradient of expected return as an *expectation* we can estimate with samples:
```
∇_θ J(θ) = E_π [ ∇_θ log π_θ(a_t|s_t) · Φ_t ]
```
where Φ_t is any valid "weight" — total return, reward-to-go, or advantage (all valid per Spinning Up).

In code, this becomes:
```python
loss = -(log_prob * weight).mean()   # negative because we maximize, optimizer minimizes
```

**TASK:**
1. `PolicyNet`: one-hot(4) → Linear(4→16) → ReLU → Linear(16→2) → logits
2. `select_action(net, state)` → `(action, log_prob)` using `Categorical(logits=logits)`
3. `reinforce_loss(log_probs, weights)` → scalar loss using the formula above
4. Run 500 training episodes on GridWorld. Average episode length should drop from ~10 to ~3.

**PREDICT (say this out loud):**
- The loss function `-(log_prob * G_t).mean()` — why is this NOT a typical supervised loss? What's different? (Spinning Up calls this out explicitly)
- If all rewards in an episode are positive (as in our GridWorld), what problem does REINFORCE have even before normalization?
- Why does the log-derivative trick require that actions are *sampled* from the policy, not taken greedily?

In [ ]:
class PolicyNet(nn.Module):
    def __init__(self, n_states=4, n_actions=2, hidden=16):
        super().__init__()
        # YOUR CODE HERE

    def forward(self, x):
        # YOUR CODE HERE — return logits
        pass

def select_action(net, state, n_states=4):
    """One-hot encode state → forward → Categorical sample. Returns (action int, log_prob tensor)."""
    x = torch.zeros(n_states); x[state] = 1.0
    # YOUR CODE HERE
    pass

def reinforce_loss(log_probs, weights):
    """
    log_probs: list of scalar tensors  (log π(a_t|s_t), differentiable)
    weights:   list or tensor of floats (Φ_t — can be G_t, reward-to-go, or advantage)
    Returns: scalar loss (negative policy gradient objective)
    """
    # YOUR CODE HERE
    # loss = -(torch.stack(log_probs) * weights_tensor).mean()
    pass


# --- Training loop ---
torch.manual_seed(0)
net = PolicyNet()
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)
lengths = []

for ep in range(500):
    s, log_probs, rewards = 0, [], []
    for _ in range(50):
        a, lp = select_action(net, s)
        s_next, r = gridworld_step(s, a)
        log_probs.append(lp); rewards.append(r)
        s = s_next
        if s == 3: break

    lengths.append(len(rewards))

    # Use reward-to-go as weight (better than total return — per Spinning Up)
    returns = torch.tensor(monte_carlo_returns(rewards), dtype=torch.float32)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)   # normalize

    loss = reinforce_loss(log_probs, returns)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if (ep + 1) % 100 == 0:
        print(f"ep={ep+1:4d}  avg_len={np.mean(lengths[-100:]):.1f}")

## Q4 — Baseline subtraction and the advantage function

**Appears in:** "What are baseline functions in policy gradient methods? Why do they work (mathematically)?" (wecreateproblems, Spinning Up). "Explain the EGLP (Expected Grad-Log-Prob) lemma and why it justifies baseline subtraction." (Spinning Up — **verbatim**). "What is the Advantage function A(s,a)? How is it computed?" (wecreateproblems, index.dev).

**Why it matters:**
This is the core variance-reduction technique in all of modern RL. The EGLP lemma (Spinning Up's term) proves you can subtract *any* function of state from the policy gradient weight without changing the expected gradient — but it *does* reduce variance. The optimal baseline is V(s). That gives you the **advantage**:
```
A(s, a) = Q(s, a) − V(s)
```
- A > 0: this action is *better than average* from this state → increase its probability
- A < 0: this action is *worse than average* → decrease its probability

**The EGLP lemma (prove this if asked):**
```
E_{a~π} [ ∇_θ log π(a|s) · b(s) ] = b(s) · E_{a~π} [∇_θ log π(a|s)]
                                    = b(s) · ∇_θ Σ_a π(a|s)
                                    = b(s) · ∇_θ 1 = 0
```
So subtracting b(s) = V(s) from returns doesn't bias the gradient.

**TASK:**
1. `ValueNet`: same input as PolicyNet, output size 1 (predicts V(s))
2. `compute_advantages(rewards, values, gamma)` → `G_t - V(s_t)` for each step
3. Modify the REINFORCE update: use advantages instead of raw returns for the actor loss; add a value loss `MSE(V(s_t), G_t)` for the critic
4. Compare to Q3: run 300 episodes and show faster convergence

**PREDICT (say this out loud):**
- Verbatim Spinning Up question: "Why is a state-dependent baseline valid (does not introduce bias)?" — answer using the EGLP lemma above
- If V is perfectly trained, what does A(s, a) ≈ 0 for all actions mean about the policy?
- Scenario trap (#18): "We collected 6 robot trajectories. 5 failed (reward≈0), 1 succeeded (reward=1). Without a baseline, what does the gradient do to the 5 failed trajectories?" (The answer is counterintuitive — think carefully)

In [ ]:
class ValueNet(nn.Module):
    def __init__(self, n_states=4, hidden=16):
        super().__init__()
        # YOUR CODE HERE — same as PolicyNet but output dim = 1

    def forward(self, x):
        pass   # return scalar value estimate

def get_value(vnet, state, n_states=4):
    x = torch.zeros(n_states); x[state] = 1.0
    with torch.no_grad():
        return vnet(x).item()

def compute_advantages(rewards, values, gamma=0.9):
    """
    returns: G_t (use monte_carlo_returns)
    advantages: G_t - V(s_t)
    values: list of V(s_t) for each t
    Returns (returns tensor, advantages tensor).
    """
    returns = torch.tensor(monte_carlo_returns(rewards, gamma), dtype=torch.float32)
    values_t = torch.tensor(values, dtype=torch.float32)
    advantages = returns - values_t
    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return returns, advantages


# --- Actor-Critic training ---
torch.manual_seed(0)
actor  = PolicyNet()
critic = ValueNet()
opt    = torch.optim.Adam(list(actor.parameters()) + list(critic.parameters()), lr=0.01)
lengths_ac = []

for ep in range(300):
    s, log_probs, rewards, states = 0, [], [], []
    for _ in range(50):
        a, lp = select_action(actor, s)
        s_next, r = gridworld_step(s, a)
        log_probs.append(lp); rewards.append(r); states.append(s)
        s = s_next
        if s == 3: break

    lengths_ac.append(len(rewards))
    values = [get_value(critic, st) for st in states]
    returns, advantages = compute_advantages(rewards, values)

    # Actor loss: policy gradient with advantage weights
    actor_loss = reinforce_loss(log_probs, advantages)

    # Critic loss: MSE between value prediction and actual returns
    value_preds = torch.cat([critic(torch.zeros(4).index_fill_(0, torch.tensor(st), 1.0))
                              for st in states])
    critic_loss = F.mse_loss(value_preds.squeeze(), returns)

    loss = actor_loss + 0.5 * critic_loss
    opt.zero_grad(); loss.backward(); opt.step()

    if (ep + 1) % 100 == 0:
        print(f"ep={ep+1:3d}  avg_len={np.mean(lengths_ac[-100:]):.1f}  "
              f"(Q3 REINFORCE was: {np.mean(lengths[-100:]):.1f})")

## Q5 — On-policy vs. off-policy: Q-learning vs. SARSA

**Appears in:** "What is the difference between on-policy and off-policy learning? Give examples of each." (interviewnode, wecreateproblems, climbtheladder — **near-universal**). "What is Q-learning? Write the Q-learning update rule." (near-universal). "Compare Q-learning and SARSA. Which is on-policy and which is off-policy, and why does it matter?" (wecreateproblems).

**Why it matters:**
This is one of the most common conceptual questions in RL interviews. The distinction determines whether an algorithm can learn from replayed data (like DQN's experience replay) or must always use fresh on-policy data (like REINFORCE and PPO). For a company training LLMs with RL, this matters: off-policy algorithms can reuse old model generations, but on-policy algorithms discard them after each update.

**Q-learning (off-policy):**
```
Q(s,a) ← Q(s,a) + α [ r + γ · max_{a'} Q(s',a') − Q(s,a) ]
```
Uses `max Q` for the next state — assumes the optimal action will be taken next, regardless of what the policy actually does.

**SARSA (on-policy):**
```
Q(s,a) ← Q(s,a) + α [ r + γ · Q(s', a') − Q(s,a) ]
```
where a' is the *actual next action taken by the policy*. Evaluates the policy being followed.

**TASK:**
1. `q_learning_update(Q, s, a, r, s_next, done, alpha, gamma)` — uses `max Q(s')`
2. `sarsa_update(Q, s, a, r, s_next, a_next, done, alpha, gamma)` — uses `Q(s', a_next)`
3. Train both on GridWorld for 500 episodes with ε=0.3 exploration
4. Print final Q-tables side by side — they should converge to the same values

**PREDICT (say this out loud):**
- Q-learning converges to the *optimal* policy even under ε-greedy. SARSA converges to the *ε-greedy* policy. What's the practical difference in a cliff-walking task?
- Why can Q-learning use experience replay (old data) but SARSA cannot?
- Which would be more conservative near dangerous states — Q-learning or SARSA?

In [ ]:
def epsilon_greedy(Q, state, epsilon):
    if np.random.rand() < epsilon:
        return np.random.randint(Q.shape[1])
    return int(np.argmax(Q[state]))

def q_learning_update(Q, s, a, r, s_next, done, alpha=0.1, gamma=0.9):
    """Off-policy: bootstrap with max_a Q(s', a)."""
    # YOUR CODE HERE
    # td_target = r + gamma * Q[s_next].max() * (1 - done)
    # Q[s, a] += alpha * (td_target - Q[s, a])
    pass

def sarsa_update(Q, s, a, r, s_next, a_next, done, alpha=0.1, gamma=0.9):
    """On-policy: bootstrap with Q(s', a_next) where a_next is the actual next action."""
    # YOUR CODE HERE
    # td_target = r + gamma * Q[s_next, a_next] * (1 - done)
    # Q[s, a] += alpha * (td_target - Q[s, a])
    pass


# --- Train both ---
np.random.seed(0)
Q_ql   = np.zeros((4, 2))
Q_sarsa = np.zeros((4, 2))

for ep in range(500):
    # Q-learning episode
    s = 0
    for _ in range(50):
        a = epsilon_greedy(Q_ql, s, 0.3)
        s_next, r = gridworld_step(s, a)
        done = (s_next == 3)
        q_learning_update(Q_ql, s, a, r, s_next, done)
        s = s_next
        if done: break

    # SARSA episode
    s = 0
    a = epsilon_greedy(Q_sarsa, s, 0.3)
    for _ in range(50):
        s_next, r = gridworld_step(s, a)
        done = (s_next == 3)
        a_next = epsilon_greedy(Q_sarsa, s_next, 0.3)
        sarsa_update(Q_sarsa, s, a, r, s_next, a_next, done)
        s, a = s_next, a_next
        if done: break

print(f"{'State':>6}  {'QL Q(left)':>10}  {'QL Q(right)':>11}  {'SARSA Q(left)':>13}  {'SARSA Q(right)':>14}")
print("-" * 62)
for s in range(4):
    print(f"{s:6d}  {Q_ql[s,0]:10.4f}  {Q_ql[s,1]:11.4f}  {Q_sarsa[s,0]:13.4f}  {Q_sarsa[s,1]:14.4f}")
print()
print("Both should converge to same V* ≈ [0.729, 0.81, 0.9, 0.0]")
print(f"Q-learning V*: {np.max(Q_ql, axis=1).round(3)}")
print(f"SARSA      V*: {np.max(Q_sarsa, axis=1).round(3)}")

## Q6 — Generalized Advantage Estimation (GAE)

**Appears in:** "What is Generalized Advantage Estimation (GAE)? What does the λ parameter control?" (wecreateproblems, Spinning Up). "Implement GAE given rewards, values, and λ." (wecreateproblems — implementation question). Listed as a research-level question across all sources, and used verbatim in PPO implementations.

**Why it matters:**
GAE is the advantage estimator used in every production PPO implementation (OpenAI Baselines, Stable-Baselines3, CleanRL). The λ parameter is a continuous knob between TD(0) (λ=0, low variance but biased) and Monte Carlo (λ=1, unbiased but high variance). Understanding GAE is a signal that you've read actual research code, not just tutorials.

**The formula:**
Define the TD residual at each step (also called the 1-step advantage):
```
δ_t = r_t + γ · V(s_{t+1}) − V(s_t)
```
Then GAE is the exponentially-weighted sum of TD residuals:
```
A_t^{GAE} = δ_t + (γλ)·δ_{t+1} + (γλ)²·δ_{t+2} + ...
           = Σ_{l≥0} (γλ)^l · δ_{t+l}
```
**Efficient implementation:** backward recurrence (same trick as computing returns):
```
A_t = δ_t + γλ · A_{t+1}
```

**TASK:**
1. `gae(rewards, values, gamma, lam)`:
   - `values` has length T+1 (includes bootstrap value for the next state after the last step)
   - compute δ_t = r_t + γ·V_{t+1} - V_t for each t
   - use backward recurrence to compute A_t
   - return advantages tensor of length T
2. Show the λ interpolation: compute GAE for the same episode with λ∈{0, 0.5, 0.95, 1.0} and print advantages. λ=0 should equal TD(0) advantages; λ=1 should equal MC - V(s).

**PREDICT (say this out loud):**
- GAE with λ=0: what does A_t reduce to? (just δ_t — one-step TD residual)
- GAE with λ=1: what does A_t reduce to? (MC return G_t minus V(s_t) — full MC advantage)
- Why does PPO typically use λ=0.95 instead of 0 or 1?

In [ ]:
def gae(rewards, values, gamma=0.9, lam=0.95):
    """
    Generalized Advantage Estimation.
    rewards: list of length T
    values:  list of length T+1  (values[T] = bootstrap value, = 0 if terminal)
    Returns: advantages tensor of length T
    """
    T = len(rewards)
    advantages = np.zeros(T)
    # YOUR CODE HERE
    # Step 1: compute delta_t = rewards[t] + gamma * values[t+1] - values[t]  for each t
    # Step 2: backward recurrence: A[T-1] = delta[T-1], then A[t] = delta[t] + gamma*lam*A[t+1]
    pass

# --- Verification episode ---
rewards_ep = [0., 0., 0., 1.]
V_perfect  = [0.729, 0.81, 0.9, 0.0, 0.0]   # V*(s), length T+1

mc_returns = np.array(monte_carlo_returns(rewards_ep))
mc_adv     = mc_returns - np.array(V_perfect[:4])    # MC advantage = G_t - V(s_t)

print(f"{'t':>3}  {'r':>5}  {'V(s)':>6}  {'lam=0 (TD)':>11}  {'lam=0.5':>8}  {'lam=0.95':>9}  {'lam=1.0 (MC)':>12}  {'MC-V(s)':>9}")
print("-" * 70)
for lam in [0, 0.5, 0.95, 1.0]:
    adv = gae(rewards_ep, V_perfect, gamma=0.9, lam=lam)
    if lam == 0:
        rows = [(t, rewards_ep[t], V_perfect[t], adv[t]) for t in range(4)]
    else:
        for t in range(4):
            rows[t] = rows[t] + (adv[t],)

for t in range(4):
    r, v = rewards_ep[t], V_perfect[t]
    advs = [gae(rewards_ep, V_perfect, gamma=0.9, lam=l)[t] for l in [0, 0.5, 0.95, 1.0]]
    print(f"{t:3d}  {r:5.1f}  {v:6.3f}  {advs[0]:11.4f}  {advs[1]:8.4f}  {advs[2]:9.4f}  {advs[3]:12.4f}  {mc_adv[t]:9.4f}")

print()
print("λ=1.0 column should equal MC-V(s) column (last two columns identical)")

## Q7 — PPO: the full loss function (clip + value + entropy)

**Appears in:** "What is PPO? How does it address instability in vanilla policy gradient methods?" (wecreateproblems, index.dev — **high frequency**). "Write out the PPO clipped surrogate objective. What is the role of ε?" (Spinning Up — verbatim). "What is the full PPO loss function? What are its three components?" (Spinning Up — verbatim). "How does PPO differ from TRPO? What computational advantage does PPO have?" (wecreateproblems, Spinning Up).

**Why it matters:**
PPO is the algorithm behind InstructGPT, ChatGPT's RLHF phase, and most LLM alignment pipelines. Spinning Up dedicates a full page to its exact loss function — the interview will test whether you've read it.

**The full PPO loss (three terms — memorize all three):**
```
L^{CLIP}(θ) = E_t [ min( r_t(θ)·A_t,  clip(r_t(θ), 1-ε, 1+ε)·A_t ) ]

L^{VF}(θ)  = E_t [ (V_θ(s_t) - V_t^{targ})² ]

L^{S}(θ)   = E_t [ H[π_θ(·|s_t)] ]     ← entropy bonus (exploration)

L^{PPO}    = L^{CLIP} − c₁·L^{VF} + c₂·L^{S}
```
where `r_t(θ) = π_θ(a_t|s_t) / π_{old}(a_t|s_t)` is the probability ratio.

**TASK:**
1. `ppo_clip_loss(log_probs_new, log_probs_old, advantages, epsilon=0.2)` → scalar
2. `entropy_loss(logits)` → scalar (mean entropy H = -Σ p·log p over the batch)
3. `ppo_total_loss(log_probs_new, log_probs_old, advantages, value_preds, value_targets, logits, epsilon, c1, c2)` → combine all three

Verify with a hand-computed example: `r=[0.5, 1.5, 2.0]`, `A=[1.0, 1.0, -1.0]`, ε=0.2

**PREDICT (say this out loud):**
- For `r=1.5, A=+1.0`: unclipped=1.5, clipped=1.2 → min=1.2. Why does the min enforce the constraint?
- For `r=2.0, A=-1.0`: what is the clipped value? Does clipping activate? (work out by hand)
- Why does PPO need an entropy bonus? What happens if you remove c₂?

In [ ]:
def ppo_clip_loss(log_probs_new, log_probs_old, advantages, epsilon=0.2):
    """
    PPO clipped surrogate objective (negated — to be minimized).
    log_probs_new: [T] differentiable
    log_probs_old: [T] detached
    advantages:    [T]
    Returns scalar loss.
    """
    # YOUR CODE HERE
    # ratio = exp(log_probs_new - log_probs_old)
    # unclipped = ratio * advantages
    # clipped   = clip(ratio, 1-eps, 1+eps) * advantages
    # loss = -mean(min(unclipped, clipped))
    pass

def entropy_bonus(logits):
    """
    Mean entropy across a batch: H = -Σ p·log(p).
    logits: [T, n_actions] or [n_actions]
    Returns negative entropy (to be subtracted from total loss — we want to maximize entropy).
    """
    # YOUR CODE HERE
    # probs = softmax(logits), then H = -(probs * log_softmax(logits)).sum(-1).mean()
    # return -H  (negated so optimizer minimizes → maximizes entropy)
    pass

def ppo_total_loss(log_probs_new, log_probs_old, advantages,
                   value_preds, value_targets, logits,
                   epsilon=0.2, c1=0.5, c2=0.01):
    """
    Full PPO loss: L_CLIP - c1*L_VF + c2*L_S  (to minimize)
    Note: L_S = -entropy, so subtracting it means maximizing entropy.
    """
    # YOUR CODE HERE
    # l_clip = ppo_clip_loss(...)
    # l_vf   = F.mse_loss(value_preds, value_targets)
    # l_ent  = entropy_bonus(logits)   ← already negated
    # return l_clip + c1 * l_vf + c2 * l_ent
    pass


# --- Hand-computed verification ---
ratios = torch.tensor([0.5, 1.5, 2.0])
advantages = torch.tensor([1.0, 1.0, -1.0])
log_probs_old = torch.zeros(3)
log_probs_new = torch.log(ratios).requires_grad_(True)

loss = ppo_clip_loss(log_probs_new, log_probs_old, advantages)
print(f"PPO clip loss: {loss.item():.4f}")
print()
print("Per-step manual verification (ε=0.2):")
for i, (r, A) in enumerate(zip([0.5, 1.5, 2.0], [1.0, 1.0, -1.0])):
    unclipped = r * A
    clipped   = np.clip(r, 1-0.2, 1+0.2) * A
    chosen    = min(unclipped, clipped)
    print(f"  r={r:.1f} A={A:+.1f}: unclipped={unclipped:.2f}  clipped={clipped:.2f}  min={chosen:.2f}")
print(f"  Expected loss = -{np.mean([min(r*A, np.clip(r,0.8,1.2)*A) for r,A in zip([0.5,1.5,2.0],[1.0,1.0,-1.0])]):.4f}")

## Q8 — Exploration vs. exploitation: entropy, epsilon-decay, and the cold-start trap

**Appears in:** "What is the exploration-exploitation trade-off? What are three advanced strategies beyond ε-greedy?" (interviewnode, wecreateproblems, index.dev — **near-universal**). "Compare ε-greedy, softmax/Boltzmann, and UCB exploration strategies." (interviewnode). Scenario trap #15: "Why is ε-greedy mathematically doomed for a robot arm in continuous high-dimensional space?" (aiinterviewprep.substack.com).

**Why it matters:**
This is the most common RL follow-up question after "what is RL?" Every interviewer asks it. For formal math RL, a theorem prover that only exploits what it knows will cycle through the same wrong proof attempts forever. Exploration is how it discovers novel proof strategies.

**Three exploration mechanisms (know all three):**

1. **ε-greedy**: random action with prob ε. Simple. Bad for high-dimensional continuous spaces.
2. **Entropy bonus**: add `β·H(π)` to reward. Policy learns to stay spread out. Used in SAC, PPO.
3. **Boltzmann/temperature**: `π(a|s) ∝ exp(Q(s,a)/τ)`. τ→0: greedy. τ→∞: uniform.

**TASK:**
1. `entropy(logits)` → scalar H(π) for a single distribution
2. `boltzmann_probs(q_values, tau)` → softmax with temperature scaling
3. `epsilon_decay_schedule(episode, total_episodes, eps_start=1.0, eps_end=0.05)` → linearly decayed ε

Then demonstrate the cold-start trap from the scenario question:
- Show that ε-greedy on a 100-dimensional action space finds reward <1% of the time after 1000 random steps
- Show entropy bonus keeps exploring while ε-greedy collapses to greedy after decay

**PREDICT (say this out loud):**
- Scenario trap: "In a 14-DoF robot arm with continuous action space, why does argmax(Q) shatter a 5ms latency budget, and how do you fix it?" (hint: the fix is having an Actor network)
- Max entropy for a 2-action policy = log(2) ≈ 0.693. Max entropy for a 100-action policy = ?
- Why does entropy bonus work better than ε-greedy for LLM token generation (hint: token vocab size)

In [ ]:
def entropy(logits):
    """H(π) = -Σ p·log(p). logits: tensor [n_actions] → scalar."""
    # YOUR CODE HERE
    pass

def boltzmann_probs(q_values, tau):
    """Temperature-scaled softmax: softmax(q / tau). q_values: tensor, tau: float."""
    # YOUR CODE HERE
    pass

def epsilon_decay_schedule(episode, total_episodes, eps_start=1.0, eps_end=0.05):
    """Linear decay from eps_start to eps_end over total_episodes."""
    # YOUR CODE HERE
    pass


# --- Verify entropy ---
uniform = torch.zeros(4)      # uniform over 4 actions: max entropy = log(4)
greedy  = torch.tensor([10., -10., -10., -10.])   # near-deterministic: H ≈ 0

print(f"Entropy of uniform(4):    {entropy(uniform):.4f}  (expected: {np.log(4):.4f})")
print(f"Entropy of near-greedy:   {entropy(greedy):.4f}   (expected: ≈ 0)")
print(f"Max entropy 100 actions:  {np.log(100):.4f}  (LLM vocab ~50k: {np.log(50000):.4f})")
print()

# --- Boltzmann temperature sweep ---
q = torch.tensor([2., 1., 0.5, 0.1])
print("Boltzmann temperature sweep on Q=[2, 1, 0.5, 0.1]:")
for tau in [0.1, 0.5, 1.0, 5.0]:
    p = boltzmann_probs(q, tau)
    print(f"  tau={tau:.1f}: probs={p.numpy().round(3)}  entropy={entropy(torch.log(p+1e-8)):.3f}")
print()

# --- Epsilon decay schedule ---
total = 500
eps_vals = [epsilon_decay_schedule(ep, total) for ep in range(0, total+1, 100)]
print("ε-decay schedule (linear, 500 episodes):")
for ep, eps in zip(range(0, total+1, 100), eps_vals):
    print(f"  episode {ep:4d}: ε={eps:.3f}")
print()

# --- Cold-start trap demo ---
np.random.seed(0)
n_actions = 100  # simulate large discrete space
n_episodes = 500
hits_eps_greedy = 0
for ep in range(n_episodes):
    eps = epsilon_decay_schedule(ep, n_episodes)
    # With decayed epsilon, mostly greedy (action 0 = learned best so far)
    action = np.random.randint(n_actions) if np.random.rand() < eps else 0
    # Reward only if lucky action (action 42 = correct)
    if action == 42: hits_eps_greedy += 1
print(f"ε-greedy hit rate in {n_actions}-action space: {hits_eps_greedy/n_episodes:.3f}")
print("(should be very low — ε-greedy locks in early, misses correct action)")

## Q9 — KL divergence and the RLHF reference policy constraint

**Appears in:** "How does PPO optimize an LLM using reward signals? What is the role of the KL penalty to the reference policy?" (dev.to RLHF series — now standard at LLM-focused companies). "Explain the full RLHF pipeline end-to-end." (dev.to RLHF series). "Why does PPO use early stopping based on KL divergence?" (Spinning Up — verbatim).

**Why it matters:**
For this company (RL for formal math with LLMs), the RLHF pipeline is directly relevant. The KL constraint prevents the RL-finetuned model from drifting so far from the base model that it stops generating coherent mathematics. Without it, the model learns to hack the reward function — outputting symbol sequences that score high but aren't valid proofs.

**The RLHF objective:**
```
reward_total(x, y) = reward_model(x, y) − β · KL(π_θ(·|x) || π_ref(·|x))
```
- `reward_model(x, y)`: learned reward signal (from human preferences or formal verification)
- `π_ref`: the original pretrained model (frozen)
- `β`: how tightly we constrain drift from the base model
- `KL(p||q) = Σ p(a) · log(p(a)/q(a))` — always ≥ 0, = 0 only when p = q

**TASK:**
1. `kl_divergence(p_logits, q_logits)` → scalar KL(p||q) using log-softmax
2. For your trained PolicyNet (from Q3), compute KL from a uniform reference policy at each state
3. Show: KL(p||p) = 0; KL is asymmetric (KL(p||q) ≠ KL(q||p))
4. Compute the RLHF-style penalized reward: `r_penalized = r_task - beta * KL(policy || ref)` for β ∈ {0, 0.1, 0.5}

**PREDICT (say this out loud):**
- RLHF regression trap: "In the first few PPO iterations of LLM finetuning, benchmarks tank. Why?" (hint: KL penalty pushes policy out of distribution before the reward model can pull it back)
- As β → ∞, what does the optimal policy converge to?
- Why is KL(π_θ || π_ref) asymmetric? What would KL(π_ref || π_θ) penalize differently?

In [ ]:
def kl_divergence(p_logits, q_logits):
    """
    KL(p || q) = Σ p(a) · [log p(a) - log q(a)]
    p_logits, q_logits: tensors [n_actions]
    Returns scalar.
    """
    # YOUR CODE HERE
    # p = softmax(p_logits)
    # KL = (p * (log_softmax(p_logits) - log_softmax(q_logits))).sum()
    pass


# --- Basic properties ---
p = torch.tensor([0.9, 0.1])
q = torch.tensor([0.5, 0.5])
p_logits = torch.log(p)
q_logits = torch.log(q)

kl_pq = kl_divergence(p_logits, q_logits)
kl_qp = kl_divergence(q_logits, p_logits)
kl_pp = kl_divergence(p_logits, p_logits)

print(f"KL(p||q) = {kl_pq:.4f}   (p=[0.9,0.1], q=[0.5,0.5])")
print(f"KL(q||p) = {kl_qp:.4f}   (asymmetric — different!)")
print(f"KL(p||p) = {kl_pp:.6f}  (should be exactly 0)")
print()

# --- Trained policy vs uniform reference ---
uniform_logits = torch.zeros(2)   # uniform over 2 actions
print("KL(trained_policy || uniform_ref) at each state:")
with torch.no_grad():
    for s in range(4):
        x = torch.zeros(4); x[s] = 1.0
        policy_logits = net(x)   # net from Q3
        kl = kl_divergence(policy_logits, uniform_logits)
        probs = F.softmax(policy_logits, dim=-1)
        print(f"  state {s}: probs={probs.numpy().round(3)}  KL={kl:.4f}")
print()

# --- RLHF penalized reward ---
r_task = 1.0   # environment reward for reaching goal
print("RLHF penalized reward at state 0 (highest KL state):")
with torch.no_grad():
    x = torch.zeros(4); x[0] = 1.0
    policy_logits = net(x)
    kl = kl_divergence(policy_logits, uniform_logits).item()
    for beta in [0.0, 0.1, 0.5, 2.0]:
        r_pen = r_task - beta * kl
        print(f"  beta={beta:.1f}: r_penalized = {r_task:.2f} - {beta:.1f}*{kl:.3f} = {r_pen:.4f}")

## Q10 — Reward design: outcome vs. process rewards and reward hacking

**Appears in:** "What is reward hacking, and why is it dangerous?" (interviewnode, wecreateproblems — **high frequency**). "What are the challenges of sparse rewards in RL? How do you handle them?" (wecreateproblems). Scenario trap #19: "You're building a reasoning model like DeepSeek R1. How do you formulate the RL objective for adaptive compute?" Scenario trap #20: "You're grading student-coded video games. How do you design the reward to catch the most bugs?" (aiinterviewprep.substack.com). Scenario trap #25: "You dramatically scale RFT sampling to 100x. Why does test error spike?" (aiinterviewprep.substack.com).

**Why it matters:**
This is the exact research problem this company works on every day. Training a language model to produce formal math proofs requires a reward signal — but what signal? Every choice has failure modes. Knowing the tradeoffs between outcome rewards (ORM) and process rewards (PRM) is the most directly relevant thing you can demonstrate in this interview.

**The landscape:**

| Approach | Reward signal | Pros | Cons |
|---|---|---|---|
| **ORM** (outcome) | +1 if final proof correct | Simple, no labeling | Sparse, no signal for near-misses |
| **PRM** (process) | +score per proof step | Dense, fast learning | Expensive to label, can be gamed |
| **Shaped** | PRM + ORM bonus | Best of both | Most complex |
| **RLHF reward model** | Learned from preferences | Flexible | Reward hacking risk |

**TASK:**
1. `outcome_reward(steps)` → +1.0 if all steps correct, else 0.0
2. `process_reward(steps, step_weight=0.2)` → per-step rewards list
3. `shaped_reward(steps, step_weight=0.2, outcome_bonus=1.0)` → combined
4. Demonstrate **reward hacking**: implement a `gaming_agent` that gets high shaped reward without actually solving the problem — by producing many short, locally-correct steps that avoid the hard final step

**PREDICT (say this out loud — these are exact scenario trap questions):**
- Trap #19: "A reasoning model should think long for hard problems, instantly for '2+2'. How do you bake this into the RL objective?" (answer: include step count in the reward)
- Trap #25: "You scale RFT from 10 to 1000 samples per problem. Why does test error spike?" (answer: model learns shortcuts specific to lucky rollouts — sample quality degrades)
- Trap #20: "You reward an agent for maximizing game score in a student's buggy Breakout. What happens?" (answer: the agent finds safe paths that avoid the bugs entirely instead of triggering them)

In [ ]:
def outcome_reward(steps):
    """steps: list of 1s (correct) and 0s (wrong). Returns +1.0 if all correct, else 0."""
    # YOUR CODE HERE
    pass

def process_reward(steps, step_weight=0.2):
    """Returns list of per-step rewards: step_weight if correct, 0.0 if wrong."""
    # YOUR CODE HERE
    pass

def shaped_reward(steps, step_weight=0.2, outcome_bonus=1.0):
    """
    Combines process + outcome: per-step rewards, plus outcome_bonus added to
    the LAST step's reward if the full trajectory is correct.
    Returns list of rewards.
    """
    # YOUR CODE HERE
    pass


# --- Four trajectories ---
trajectories = {
    "perfect (5/5)":    [1, 1, 1, 1, 1],
    "one mistake":      [1, 1, 1, 0, 1],
    "all wrong":        [0, 0, 0, 0, 0],
    "hacked (partial)": [1, 1, 0, 0, 0],   # gets partial credit, avoids hard steps
}

print(f"{'Trajectory':<20}  {'ORM':>5}  {'PRM total':>10}  {'Shaped total':>13}")
print("-" * 55)
for name, steps in trajectories.items():
    orm = outcome_reward(steps)
    prm = sum(process_reward(steps))
    shp = sum(shaped_reward(steps))
    print(f"{name:<20}  {orm:5.1f}  {prm:10.2f}  {shp:13.2f}")

print()
print("=== Reward Hacking Demo ===")
print("A gaming agent produces 20 trivially-correct steps to avoid the hard final step:")
gaming_steps = [1] * 20 + [0]   # 20 correct trivial steps, fails on the hard one
print(f"  ORM reward:    {outcome_reward(gaming_steps):.1f}  (correctly penalized)")
print(f"  PRM reward:    {sum(process_reward(gaming_steps)):.2f}  (HACKED — high despite failure!)")
print(f"  Shaped reward: {sum(shaped_reward(gaming_steps)):.2f}")
print()
print("Fix: add a LENGTH PENALTY to shaped reward to discourage unnecessary steps.")
print("  r_penalized = shaped_reward - alpha * len(steps)")
for alpha in [0.0, 0.05, 0.1]:
    r = sum(shaped_reward(gaming_steps)) - alpha * len(gaming_steps)
    print(f"  alpha={alpha}: total = {r:.2f}")